# Agent: Camera (Phase 4)

This notebook subscribes to person state updates and publishes allow/deny camera decisions.

Phase 4 scope:
- Subscribe to `simulated-city/stadium/person/state`
- Publish to `simulated-city/stadium/camera/decision`
- Use random policy with more allowed than denied
- Include false positives and false negatives
- Emit fixed `reason_code` taxonomy

In [ ]:
# Cell purpose: import modules and load configuration for camera agent
from __future__ import annotations

from datetime import datetime, timezone
import json
import random

from simulated_city.config import load_config
import simulated_city.mqtt as mqtt

cfg = load_config()
sim_cfg = cfg.simulation
mqtt_cfg = cfg.mqtt

print(f"Loaded config. MQTT base topic: {mqtt_cfg.base_topic}")
print(f"Simulation section present: {sim_cfg is not None}")

In [ ]:
# Cell purpose: read decision parameters from config and define reason-code taxonomy
TRUE_ALLOW_PROBABILITY = sim_cfg.true_allow_probability if sim_cfg else 0.80
FALSE_POSITIVE_RATE = sim_cfg.false_positive_rate if sim_cfg else 0.05
FALSE_NEGATIVE_RATE = sim_cfg.false_negative_rate if sim_cfg else 0.10
CONFIDENCE_THRESHOLD = sim_cfg.camera_confidence_threshold if sim_cfg else 0.70
SEED = sim_cfg.seed if sim_cfg and sim_cfg.seed is not None else 42

REASON_CODES = [
    "allowed_lucky",
    "allowed_clear",
    "denied_unlucky",
    "denied_flagged",
    "camera_false_positive",
    "camera_false_negative",
]

rng = random.Random(SEED + 100)

print(f"Decision settings -> allow={TRUE_ALLOW_PROBABILITY}, fp={FALSE_POSITIVE_RATE}, fn={FALSE_NEGATIVE_RATE}, confidence_threshold={CONFIDENCE_THRESHOLD}")
print(f"Reason-code taxonomy size: {len(REASON_CODES)}")

In [ ]:
# Cell purpose: connect to MQTT and define Phase 4 topics
client = mqtt.connect_mqtt(mqtt_cfg, client_id_suffix="agent-camera-phase4")

person_state_topic = f"{mqtt_cfg.base_topic}/person/state"
camera_decision_topic = f"{mqtt_cfg.base_topic}/camera/decision"

print(f"Connected to MQTT broker at {mqtt_cfg.host}:{mqtt_cfg.port}")
print(f"Subscribing to: {person_state_topic}")
print(f"Publishing decisions to: {camera_decision_topic}")

In [ ]:
# Cell purpose: define callback that consumes person state and publishes camera decisions
decision_count = 0
publish_success = 0
publish_fail = 0
seen_people = set()

def build_decision(person_payload: dict) -> dict:
    true_allowed = rng.random() < TRUE_ALLOW_PROBABILITY
    observed_allowed = true_allowed
    confidence = round(rng.uniform(0.60, 0.99), 3)

    false_negative_applied = False
    false_positive_applied = False

    if true_allowed and rng.random() < FALSE_NEGATIVE_RATE:
        observed_allowed = False
        false_negative_applied = True
    elif (not true_allowed) and rng.random() < FALSE_POSITIVE_RATE:
        observed_allowed = True
        false_positive_applied = True

    if false_positive_applied:
        reason_code = "camera_false_positive"
    elif false_negative_applied:
        reason_code = "camera_false_negative"
    elif observed_allowed and confidence >= CONFIDENCE_THRESHOLD:
        reason_code = "allowed_clear"
    elif observed_allowed:
        reason_code = "allowed_lucky"
    elif confidence >= CONFIDENCE_THRESHOLD:
        reason_code = "denied_flagged"
    else:
        reason_code = "denied_unlucky"

    return {
        "person_id": person_payload.get("person_id", "unknown"),
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "allowed": bool(observed_allowed),
        "reason_code": reason_code,
        "confidence": confidence,
        "entry_id": person_payload.get("target_entry_id", "unknown"),
    }

def on_message(client, userdata, msg):
    global decision_count, publish_success, publish_fail

    try:
        payload = json.loads(msg.payload.decode())
    except Exception:
        return

    person_id = payload.get("person_id")
    state = payload.get("state")
    if not person_id or state != "approaching_entry":
        return

    if person_id in seen_people:
        return

    seen_people.add(person_id)
    decision_payload = build_decision(payload)

    ok = mqtt.publish_json_checked(client, camera_decision_topic, decision_payload, qos=0)
    decision_count += 1
    if ok:
        publish_success += 1
    else:
        publish_fail += 1

    print(
        f"decision#{decision_count} person={decision_payload['person_id']} allowed={decision_payload['allowed']} "
        f"reason={decision_payload['reason_code']} confidence={decision_payload['confidence']} "
        f"publish_ok={ok}"
    )

client.on_message = on_message
client.subscribe(person_state_topic, qos=0)

print("Camera callback registered and subscription active.")

In [ ]:
# Cell purpose: keep the camera agent running and print periodic counters
import time

print("Camera agent running. Press Interrupt to stop.")
while True:
    time.sleep(5)
    print(
        f"status decisions={decision_count} success={publish_success} fail={publish_fail} tracked_people={len(seen_people)}"
    )